In [5]:
"""
Wheeler–Kiladis style wavenumber–frequency analysis for daily lat/lon/time data.

Features
- Accepts an xarray.DataArray (time, lat, lon) in daily cadence.
- Averages over a latitude band [lat_south, lat_north].
- Forms **symmetric** and **antisymmetric** components about the equator using paired
  latitudes ±φ (cosine-weighted), then averages each across the band.
- Splits the time dimension into `n_segments` pieces of length `ndays` (drops remainder).
- Computes 2D power spectra in (time, lon) with cosine tapers and averages over segments.
- Creates **ratio plots**: power divided by a smoothed 2D background (box filter).
- Overlays equatorial shallow-water dispersion curves for equivalent depths h = 12, 25, 50 m
  (Kelvin, n=1–3 Rossby, n=1–3 EIG/WIG, and MRG) using standard Matsuno (1966) relations.


Notes
-----
- The input must be daily; if not, resample to 1D with mean prior to calling.
- Longitude must be regular; 0–360 or -180–180 are both fine.
- Units do not matter for the spectrum shape; many users detrend or remove seasonal
  cycles upstream if desired.
"""
from __future__ import annotations

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter
from dataclasses import dataclass

# -------------------------------
# Physical constants for curves
# -------------------------------
Omega = 7.2921159e-5           # Earth's rotation (s^-1)
a_earth = 6_371_000.0          # Earth radius (m)
g = 9.80665                    # gravity (m s^-2)


file_in = '/glade/work/rneale/data/NOAA/NOAA_dmeans_ts_FLUT.nc'

da = xr.open_dataset(file_in)['FLUT']  # (time, lat, lon)
wk = WKAnalyzer(da, lat_south=-5, lat_north=5, ndays=96, n_segments=None)
results = wk.run()
wk.plot_symmetric(results)
wk.plot_antisymmetric(results)

IndexError: Boolean array size 201 is used to index array with shape (14459, 1, 144).

In [2]:
# -------------------------------------------------
# Utility: ensure longitude is increasing and regular
# -------------------------------------------------

def _standardize_lon(da: xr.DataArray) -> xr.DataArray:
    lon = da["lon"]
    # Wrap to [0, 360)
    lon2 = ((lon % 360) + 360) % 360
    da2 = da.assign_coords(lon=lon2).sortby("lon")
    # Ensure equally spaced
    dlon = np.diff(da2["lon"].values)
    if not np.allclose(dlon, dlon[0], rtol=0, atol=1e-6):
        raise ValueError("Longitude grid must be regular.")
    return da2

# -------------------------------------------------
# Symmetric/antisymmetric components about equator
# -------------------------------------------------

def _sym_asym_band_mean(da: xr.DataArray, lat_south: float, lat_north: float) -> tuple[xr.DataArray, xr.DataArray]:
    """Return cosine-weighted band-mean of symmetric and antisymmetric parts.

    Symmetric:  (F(φ) + F(-φ)) / 2
    Antisym.:   (F(φ) - F(-φ)) / 2

    Then average each over φ ∈ [0, band] using cos(φ) weights built from paired points.
    The band may be asymmetric in the input; we internally use |φ| pairs.
    """
    if lat_south >= lat_north:
        raise ValueError("lat_south must be < lat_north")

    # Interpolate so that we can sample matched ±phi pairs
    da = da.sortby("lat")
    maxabs = max(abs(lat_south), abs(lat_north))
    # Use only latitudes present within the band intersection of both hemispheres
    phis = np.linspace(0.0, maxabs, num=201)  # dense enough for smooth average

    f_pos = da.interp(lat=(('lat', phis)))
    f_neg = da.interp(lat=(('lat', -phis)))

    sym_phi = 0.5 * (f_pos + f_neg)
    asym_phi = 0.5 * (f_pos - f_neg)

    # Restrict to requested band [lat_south, lat_north] in absolute sense
    mask = (phis >= max(0.0, abs(lat_south))) & (phis <= abs(lat_north))
    phis_band = np.deg2rad(phis[mask])

    w = np.cos(phis_band)  # cosine weights
    w = w / w.sum()

    sym_mean = (sym_phi.isel(lat=mask) * xr.DataArray(w, dims=['lat'])).sum('lat')
    asym_mean = (asym_phi.isel(lat=mask) * xr.DataArray(w, dims=['lat'])).sum('lat')

    return sym_mean, asym_mean

# -----------------------------------
# Spectral analysis core (time, lon)
# -----------------------------------

def _cosine_taper(n: int, frac: float = 0.1) -> np.ndarray:
    """Return a 1D cosine taper window of length n with edge fraction `frac`.
    frac ∈ [0, 0.5]."""
    frac = max(0.0, min(0.5, float(frac)))
    m = int(np.floor(frac * n))
    w = np.ones(n)
    if m > 0:
        x = np.linspace(0, np.pi, m)
        w[:m] = 0.5 * (1 - np.cos(x))
        w[-m:] = w[:m][::-1]
    return w

@dataclass
class WKAnalyzer:
    da: xr.DataArray  # (time, lat, lon) daily
    lat_south: float
    lat_north: float
    ndays: int = 96
    n_segments: int | None = None  # if None, inferred as floor(len(time)/ndays)
    taper_frac_time: float = 0.1
    taper_frac_lon: float = 0.1
    smooth_hw_freq: int = 3  # smoothing half-widths (box) in frequency bins
    smooth_hw_waven: int = 3  # smoothing half-widths (box) in wavenumber bins

    def _prepare(self) -> tuple[xr.DataArray, xr.DataArray]:
        da = _standardize_lon(self.da)
        if not set(["time", "lat", "lon"]).issubset(set(da.dims)):
            raise ValueError("Input DataArray must have dims (time, lat, lon)")
        # Symmetric / antisymmetric band means
        sym, asym = _sym_asym_band_mean(da, self.lat_south, self.lat_north)
        # Drop any NaN longitudes resulting from weights
        sym = sym.dropna("lon", how="any")
        asym = asym.dropna("lon", how="any")
        return sym, asym

    def _segments(self, nt: int) -> list[slice]:
        nd = int(self.ndays)
        if self.n_segments is None:
            nseg = nt // nd
        else:
            nseg = int(self.n_segments)
        if nseg < 1:
            raise ValueError("Not enough time points to form at least 1 segment")
        segs = [slice(i*nd, (i+1)*nd) for i in range(nseg)]
        return segs

    def _spectrum(self, arr: xr.DataArray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Average 2D power spectrum over segments.
        Returns (Pavg, freq_cpd, m_wavenumbers)."""
        arr = arr.load()  # ensure memory for numpy FFT
        nt = arr.sizes["time"]
        nlon = arr.sizes["lon"]
        segs = self._segments(nt)

        # Frequency vector (cycles per day) for ndays
        nd = self.ndays
        freq = np.fft.fftfreq(nd, d=1.0)  # 1/day since daily data
        # keep only non-negative frequencies for plotting
        pos = freq >= 0
        freq_pos = freq[pos]

        # Zonal wavenumbers as *integer* spectral indices (after fftshift)
        m = np.fft.fftfreq(nlon, d=1.0 / nlon)  # returns [0.., negative]
        m = np.round(m).astype(int)
        m = np.fft.fftshift(m)

        # Tapers
        wt = _cosine_taper(nd, self.taper_frac_time)
        wx = _cosine_taper(nlon, self.taper_frac_lon)
        W = np.outer(wt, wx)

        Psum = 0.0
        count = 0
        for s in segs:
            seg = arr.isel(time=s)
            if seg.sizes["time"] != nd:
                continue  # drop incomplete tail if any
            A = seg.values  # (time, lon)
            # Remove time-mean at each lon (common in WK)
            A = A - np.nanmean(A, axis=0, keepdims=True)
            # Apply tapers
            A = A * W
            # 2D FFT: time then lon
            F = np.fft.fft(A, axis=0)  # time FFT length nd
            F = F[pos, :]              # keep non-negative freqs
            F = np.fft.fftshift(np.fft.fft(F, axis=1), axes=1)  # lon FFT + shift
            P = (np.abs(F) ** 2).astype(np.float64)
            Psum = Psum + np.nanmean(P, axis=-1, keepdims=False) if False else Psum + P
            count += 1
        if count == 0:
            raise ValueError("No complete segments of length ndays were found.")

        Pavg = Psum / count  # shape (nfreq_pos, nlon)
        return Pavg, freq_pos, m

    def run(self):
        sym, asym = self._prepare()
        Psym, fcpd, m = self._spectrum(sym)
        Pasym, _, _ = self._spectrum(asym)

        # 2D background smoothing (box filter)
        def smooth2d(P):
            size = (2*self.smooth_hw_freq+1, 2*self.smooth_hw_waven+1)
            return uniform_filter(P, size=size, mode='nearest')

        Psym_bg = smooth2d(Psym)
        Pasym_bg = smooth2d(Pasym)

        # Ratio (avoid divide-by-zero)
        Psym_ratio = Psym / np.maximum(Psym_bg, 1e-12)
        Pasym_ratio = Pasym / np.maximum(Pasym_bg, 1e-12)

        return {
            'Psym_ratio': Psym_ratio,
            'Pasym_ratio': Pasym_ratio,
            'freq_cpd': fcpd,
            'm': m,
        }

    # -----------------------------------
    # Dispersion curves (Matsuno modes)
    # -----------------------------------
    def _dispersion_curves(self, mmax: int = 30):
        m = np.arange(1, mmax+1)
        k = m / a_earth  # continuous zonal wavenumber (rad/m)
        beta = 2 * Omega / a_earth
        depths = [12.0, 25.0, 50.0]

        curves = []
        for h in depths:
            c = np.sqrt(g * h)
            Ld = np.sqrt(c / beta)

            # Kelvin (eastward only): omega = c k
            w_kelvin = c * k

            # Mixed Rossby-Gravity (n=0): two branches
            # omega = 0.5*(-beta/k ± sqrt((beta/k)^2 + 4 c^2 k^2))
            bk = beta / k
            disc = np.sqrt(bk**2 + 4 * c**2 * k**2)
            w_mrg_plus = 0.5 * (-bk + disc)   # typically eastward
            w_mrg_minus = 0.5 * (-bk - disc)  # typically westward

            # Rossby (n=1..3): omega = - beta k / (k^2 + (2n+1)/Ld^2)
            rossby = {}
            for n in (1, 2, 3):
                denom = k**2 + (2*n + 1) / (Ld**2)
                rossby[n] = - beta * k / denom

            # Inertia–Gravity (n=1..3): omega = ± sqrt(c^2 k^2 + (2n+1) beta c)
            eig = {}
            wig = {}
            for n in (1, 2, 3):
                w = np.sqrt(c**2 * k**2 + (2*n + 1) * beta * c)
                eig[n] = +w
                wig[n] = -w

            curves.append({
                'h': h,
                'm': m,
                'kelvin': w_kelvin,
                'mrg_plus': w_mrg_plus,
                'mrg_minus': w_mrg_minus,
                'rossby': rossby,
                'eig': eig,
                'wig': wig,
            })
        return curves

    # -----------------
    # Plotting helpers
    # -----------------
    def _plot_ratio(self, P_ratio: np.ndarray, fcpd: np.ndarray, m: np.ndarray, title: str):
        # Build coordinate grids for pcolormesh
        M, F = np.meshgrid(m, fcpd)
        fig = plt.figure(figsize=(7, 7))  # square
        ax = fig.add_subplot(111)
        # Use symmetric wavenumber range centered at zero
        im = ax.pcolormesh(M, F, P_ratio, shading='auto')
        cb = fig.colorbar(im, ax=ax, pad=0.01)
        cb.set_label('Power / Background')
        ax.set_xlabel('Zonal wavenumber (m)')
        ax.set_ylabel('Frequency (cycles/day)')
        ax.set_title(title)
        ax.set_xlim(m.min(), m.max())
        ax.set_ylim(0, fcpd.max())

        # Overlay dispersion curves converted to cycles/day
        curves = self._dispersion_curves(mmax=int(m.max()))
        t_sec_per_day = 86400.0
        for cset, color in zip(curves, ['k', 'tab:blue', 'tab:orange']):
            mpos = cset['m']
            # Kelvin on +m
            f_kelvin = (cset['kelvin'] / (2*np.pi)) * t_sec_per_day
            ax.plot(mpos, f_kelvin, lw=1.5, color=color, label=f"Kelvin h={cset['h']} m")
            # MRG: plot both branches on ±m (mirror to negative m)
            for key in ('mrg_plus', 'mrg_minus'):
                f = (cset[key] / (2*np.pi)) * t_sec_per_day
                ax.plot(mpos, f, lw=1.0, color=color, ls='--')
                ax.plot(-mpos, f, lw=1.0, color=color, ls='--')
            # Rossby n=1..3 (mostly westward). Plot for +m then mirror to -m
            for n in (1, 2, 3):
                f = (cset['rossby'][n] / (2*np.pi)) * t_sec_per_day
                ax.plot(mpos, f, lw=1.0, color=color, ls=':')
                ax.plot(-mpos, f, lw=1.0, color=color, ls=':')
            # IG n=1..3: eastward (EIG) on ±m, and westward (WIG)
            for n in (1, 2, 3):
                f = (cset['eig'][n] / (2*np.pi)) * t_sec_per_day
                ax.plot(mpos, f, lw=1.0, color=color)
                ax.plot(-mpos, f, lw=1.0, color=color)
                f = (cset['wig'][n] / (2*np.pi)) * t_sec_per_day
                ax.plot(mpos, f, lw=1.0, color=color)
                ax.plot(-mpos, f, lw=1.0, color=color)

        ax.legend(loc='upper right', fontsize=8, frameon=True)
        fig.tight_layout()
        return fig, ax

    def plot_symmetric(self, results: dict):
        return self._plot_ratio(results['Psym_ratio'], results['freq_cpd'], results['m'],
                                'SYMMETRIC: Power / Smoothed Background')

    def plot_antisymmetric(self, results: dict):
        return self._plot_ratio(results['Pasym_ratio'], results['freq_cpd'], results['m'],
                                'ANTISYMMETRIC: Power / Smoothed Background')


# End of module